In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = next(p for p in [cwd, *cwd.parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, r2_score, roc_auc_score

from src.features.preprocessing import load_model_data
from src.evaluation import splits, leakage
from src.evaluation.make_splits import load_split, SPLITS

SPLIT_LIST = list(SPLITS)

In [3]:
# ===== Settings: change these, then Restart & Run All =====
SPLIT = "7.5km_subbasin"            # "10km", "10km_subbasin", "7.5km_subbasin", "5km_subbasin"
USE_SUBBASIN = True      # True: subbasin features replace elevation
# MODEL_SEED = 0
THRESHOLD_MG_L = 10
DROP_ELEVATION = USE_SUBBASIN


In [4]:
X, y, meta = load_model_data(subbasin=USE_SUBBASIN, drop_elevation=DROP_ELEVATION)

FEATURES = list(X.columns)

data = load_split(pd.concat([meta, X], axis=1).assign(log_nitrate=y), split=SPLIT)
print(f"split={SPLIT} | subbasin features={USE_SUBBASIN} | {len(FEATURES)} features")
print(data["fold"].value_counts().sort_index().to_dict(), "| holdout rows:", int(data["holdout"].sum()))


split=7.5km_subbasin | subbasin features=True | 59 features
{0: 944, 1: 1114, 2: 1044, 3: 1202, 4: 976} | holdout rows: 976


In [5]:
from src.features.transformers import add_subbasin

if "subbasin" not in data.columns:
    data = add_subbasin(data)          # for reporting only; not added to FEATURES

print(data.groupby("fold")["subbasin"].value_counts(normalize=True).unstack().round(2))


subbasin  180/400 FOOT AQUIFER  ATASCADERO AREA  EAST SIDE AQUIFER  \
fold                                                                 
0                         0.15             0.00               0.08   
1                         0.12             0.10               0.17   
2                         0.29             0.03               0.05   
3                         0.10             0.08               0.25   
4                         0.28             0.01               0.08   

subbasin  FOREBAY AQUIFER  LANGLEY AREA  MONTEREY  PASO ROBLES AREA  SEASIDE  \
fold                                                                           
0                    0.32           0.0      0.00              0.26      0.0   
1                    0.27           0.0      0.00              0.17      0.0   
2                    0.30           0.0      0.00              0.15      0.0   
3                    0.25           0.0      0.01              0.17      0.0   
4                    0.29    

In [6]:
# Set the training folds and the holdout

train_val = data[~data["holdout"]].reset_index(drop=True)       # Training set with 4 folds


X_tv = train_val[FEATURES]
y_tv = train_val["log_nitrate"]

# Print the sizes of the training folds
print(train_val["fold"].value_counts().sort_index().to_dict())

{0: 944, 1: 1114, 2: 1044, 3: 1202}


Here you can begin training your model.

In [7]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.base import clone
import xgboost

In [8]:
def make_model():
    """Replace with your model. A new, untrained model is created for every fold."""
    return DummyRegressor(strategy="mean")

xy = leakage.projected_xy(train_val, "gm_latitude", "gm_longitude")
oof_pred = np.full(len(train_val), np.nan)        # model prediction, out of fold
mean_pred = np.full(len(train_val), np.nan)       # baseline: training-fold mean
nearest_m = np.full(len(train_val), np.nan)       # distance to nearest training well
fold_scores = []

for train, test in splits.iter_folds(train_val["fold"]):
    model = make_model()
    model.fit(X_tv[train], y_tv[train])
    oof_pred[test] = model.predict(X_tv[test])
    mean_pred[test] = y_tv[train].mean()
    nearest_m[test] = leakage.nearest_training_distance(xy, train, test)

    fold_scores.append({
        "fold": int(train_val.loc[test, "fold"].iloc[0]),
        "n": int(test.sum()),
        "rmse_log": root_mean_squared_error(y_tv[test], oof_pred[test]),
        "rmse_mean": root_mean_squared_error(y_tv[test], mean_pred[test]),
    })

exceeds = np.expm1(y_tv) > THRESHOLD_MG_L
print(pd.DataFrame(fold_scores).round(3).to_string(index=False))
print(f"Overall RMSE (log): {root_mean_squared_error(y_tv, oof_pred):.3f} "
      f"| R²: {r2_score(y_tv, oof_pred):.3f} "
      f"| AUC (>{THRESHOLD_MG_L} mg/L): {roc_auc_score(exceeds, oof_pred):.3f}")


 fold    n  rmse_log  rmse_mean
    0  944     1.153      1.153
    1 1114     1.196      1.196
    2 1044     1.165      1.165
    3 1202     1.317      1.317
Overall RMSE (log): 1.215 | R²: -0.010 | AUC (>10 mg/L): 0.457


# Nested CV Example

In [9]:
FEATURES

['gm_well_category',
 'soil_slopegradwta',
 'soil_aws0150wta',
 'soil_hydgrpdcd_pct_a',
 'soil_hydgrpdcd_pct_b',
 'soil_hydgrpdcd_pct_c',
 'topographic_position',
 'pct_veg_berry_500m',
 'pct_vineyard_500m',
 'pct_orchard_500m',
 'pct_grain_field_pasture_500m',
 'pct_idle_fallow_500m',
 'pct_urban_500m',
 'pct_cropland_500m',
 'crop_cycles_500m',
 'pct_multicrop_500m',
 'outside_dwr_polygons_pct_500m',
 'pct_orchard_1000m',
 'pct_grain_field_pasture_1000m',
 'pct_idle_fallow_1000m',
 'pct_urban_1000m',
 'outside_dwr_polygons_pct_1000m',
 'delta_veg_berry_pp_500m',
 'delta_vineyard_pp_500m',
 'delta_orchard_pp_500m',
 'delta_grain_field_pasture_pp_500m',
 'delta_idle_fallow_pp_500m',
 'delta_cropland_pp_500m',
 'delta_multicrop_pp_500m',
 'delta_dwr_polygon_footprint_pp_500m',
 'delta_veg_berry_pp_1000m',
 'delta_vineyard_pp_1000m',
 'delta_orchard_pp_1000m',
 'delta_grain_field_pasture_pp_1000m',
 'delta_idle_fallow_pp_1000m',
 'delta_urban_pp_1000m',
 'delta_cropland_pp_1000m',
 'delt

In [10]:
SUBFOLD_SEED = 77
MODEL_SEED = 99
N_SUBFOLDS = 4
THRESHOLD_MG_L = 10

In [11]:
from src.models import random_forest
# "data" comes from load_split (see earlier template)
# this function creates the subfolds

development, X_dev, y_dev = random_forest.make_inner_subfolds(data, FEATURES, N_SUBFOLDS = N_SUBFOLDS, THRESHOLD_MG_L = THRESHOLD_MG_L, SEED = SUBFOLD_SEED)

subfold    0    1    2    3
fold                       
0        246  275  259  164
1        230  187  350  347
2        311  288  212  233
3        212  280  211  499


In [12]:
param_grid = {
    "learning_rate": [0.03, 0.05],
    "n_estimators": [20, 35, 50, 75, 100],
    "max_depth": [2, 3, 4],
    "min_child_weight": [5, 20, 50],
    "subsample": [0.5, 0.7],
}


In [13]:
# # customize this function to your estimator. Leave all other fields as is. 
# from sklearn.model_selection import GridSearchCV
# import xgboost

# def make_search_brt(inner_cv, param_grid, SEED):
#     return GridSearchCV(
#         estimator=xgboost.XGBRegressor(
#             objective="reg:squarederror",
#             random_state=SEED,
#             n_jobs=1,
#         ),
#         param_grid=param_grid,
#         scoring="neg_root_mean_squared_error",
#         cv=inner_cv,
#         refit=True,
#         n_jobs=-1,
#     )


In [14]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, brier_score_loss

def make_search_brt(inner_cv, param_grid, SEED):
    return RandomizedSearchCV(
        estimator=xgboost.XGBRegressor(objective="reg:squarederror", random_state=SEED, n_jobs=1),
        param_distributions=param_grid,
        n_iter=20,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        refit=True,
        random_state=SEED,
        n_jobs=-1,
    )


def make_search_brt_clf(inner_cv, param_grid, SEED):
    return RandomizedSearchCV(
        estimator=xgboost.XGBClassifier(objective="binary:logistic", max_bin=64,
                                        random_state=SEED, n_jobs=1),
        param_distributions=param_grid,
        n_iter=20,
        scoring="neg_brier_score",
        cv=inner_cv,
        refit=True,
        random_state=SEED,
        n_jobs=-1,
    )



In [15]:
def choose_params(outer_results, params_by_fold):
    """Most frequent parameters across outer folds; ties go to the lowest inner RMSE."""
    cols = [c for c in params_by_fold.columns if c != "outer_fold"]
    table = params_by_fold.merge(outer_results[["outer_fold", "best_inner_rmse_log"]], on="outer_fold")
    table["count"] = table.groupby(cols)["outer_fold"].transform("size")
    table = table.sort_values(["count", "best_inner_rmse_log"], ascending=[False, True])
    return table[cols].iloc[[0]].to_dict("records")[0]


In [16]:
def nested_cv_classifier(development, X_dev, exceeds_dev, param_grid, make_search, seed):
    """Nested spatial CV for a classifier: per-fold AUC and Brier skill, plus out-of-fold probabilities."""
    fold_dev = development["fold"].to_numpy()
    subfold_dev = development["subfold"].to_numpy()
    oof_prob = np.full(len(development), np.nan)
    rows = []

    for k in sorted(np.unique(fold_dev)):
        test = np.flatnonzero(fold_dev == k)
        train = np.flatnonzero(fold_dev != k)
        sub = subfold_dev[train]
        inner_cv = [(np.flatnonzero(sub != s), np.flatnonzero(sub == s)) for s in range(N_SUBFOLDS)]

        search = make_search(inner_cv, param_grid, seed)
        search.fit(X_dev[train], exceeds_dev[train])
        oof_prob[test] = search.predict_proba(X_dev[test])[:, 1]

        base_rate = exceeds_dev[train].mean()      # baseline: always predict the training exceedance rate
        brier = brier_score_loss(exceeds_dev[test], oof_prob[test])
        brier_base = brier_score_loss(exceeds_dev[test], np.full(len(test), base_rate))
        rows.append({"fold": int(k),
                     "auc": roc_auc_score(exceeds_dev[test], oof_prob[test]),
                     "brier_skill": 1 - brier / brier_base,
                     "best_inner_brier": -search.best_score_,
                     **search.best_params_})
    return pd.DataFrame(rows), oof_prob


In [17]:
CONFIGS = [                      # (split, use subbasin features)
    ("10km", False),
    ("10km", True),
    ("10km_subbasin", True),
    ("7.5km_subbasin", True),
    ("5km_subbasin", True),
]

In [18]:
import time
from src.evaluation.results_log import save_run, load_runs, summarize


def run_config(split, use_subbasin):
    drop_elevation = use_subbasin                    # subbasin replaces elevation
    X, y, meta = load_model_data(subbasin=use_subbasin, drop_elevation=drop_elevation)
    features = list(X.columns)
    d = load_split(pd.concat([meta, X], axis=1).assign(log_nitrate=y), split=split)

    development, X_dev, y_dev = random_forest.make_inner_subfolds(
        d, features, N_SUBFOLDS=N_SUBFOLDS, THRESHOLD_MG_L=THRESHOLD_MG_L, SEED=SUBFOLD_SEED)

    outer_results, params_by_fold = random_forest.fit_model_with_nested_cv(
        development, X_dev, y_dev, param_grid, make_search_brt,
        N_SUBFOLDS=N_SUBFOLDS, THRESHOLD_MG_L=THRESHOLD_MG_L, SEED=MODEL_SEED)
    best_params = choose_params(outer_results, params_by_fold)

    model = xgboost.XGBRegressor(**best_params, objective="reg:squarederror",
                                 max_bin=64, random_state=MODEL_SEED, n_jobs=-1)
    results = random_forest.fit_model_with_normal_cv(
        development, X_dev, y_dev, model, THRESHOLD_MG_L=THRESHOLD_MG_L)

    save_run(results["outer_results"], model="XGBoost BRT", task="regression", author="Karuna",
             split=split, use_subbasin=use_subbasin, drop_elevation=drop_elevation,
             n_features=len(features), notes=f"params: {best_params}")
    return {"split": split, "subbasin": use_subbasin,
            "nested_rmse_log": outer_results["rmse_log"].mean(),
            "rmse_log": results["outer_results"]["rmse_log"].mean(),
            "r2": results["outer_results"]["r2"].mean(), **best_params}

summary = []
for split, use_subbasin in CONFIGS:
    start = time.time()
    summary.append(run_config(split, use_subbasin))
    print(f"{split} | subbasin={use_subbasin} | {(time.time() - start) / 60:.1f} min")

pd.DataFrame(summary).round(3)


subfold    0    1    2    3
fold                       
0        187  195  237  416
1        109  194  230  439
2        408  258  302  175
3        186  160  224  471
Saved 4 folds to results/model_runs/karuna__xgboost-brt__regression__10km__subbasin0__noelev0.csv
10km | subbasin=False | 1.0 min
subfold    0    1    2    3
fold                       
0        187  195  237  416
1        109  194  230  439
2        408  258  302  175
3        186  160  224  471
Saved 4 folds to results/model_runs/karuna__xgboost-brt__regression__10km__subbasin1__noelev1.csv
10km | subbasin=True | 0.9 min
subfold    0    1    2    3
fold                       
0        229  224  471  194
1        314  439  183  161
2        186  262  412  131
3        302  408  158  182
Saved 4 folds to results/model_runs/karuna__xgboost-brt__regression__10km-subbasin__subbasin1__noelev1.csv
10km_subbasin | subbasin=True | 0.8 min
subfold    0    1    2    3
fold                       
0        246  275  259  164
1     

,split,subbasin,nested_rmse_log,rmse_log,r2,subsample,n_estimators,min_child_weight,max_depth,learning_rate
0,10km,False,1.084,1.071,0.238,0.5,75,50,4,0.03
1,10km,True,1.061,1.047,0.272,0.5,75,50,4,0.03
2,10km_subbasin,True,0.993,0.989,0.321,0.7,75,5,3,0.05
3,7.5km_subbasin,True,1.028,1.026,0.256,0.5,75,5,3,0.05
4,5km_subbasin,True,1.005,1.006,0.299,0.7,75,50,3,0.03


In [19]:
def run_config_clf(split, use_subbasin):
    drop_elevation = use_subbasin
    X, y, meta = load_model_data(subbasin=use_subbasin, drop_elevation=drop_elevation)
    features = list(X.columns)
    d = load_split(pd.concat([meta, X], axis=1).assign(log_nitrate=y), split=split)

    development, X_dev, y_dev = random_forest.make_inner_subfolds(
        d, features, N_SUBFOLDS=N_SUBFOLDS, THRESHOLD_MG_L=THRESHOLD_MG_L, SEED=SUBFOLD_SEED)
    exceeds_dev = (np.expm1(y_dev) > THRESHOLD_MG_L).astype(int)

    fold_scores, oof_prob = nested_cv_classifier(
        development, X_dev, exceeds_dev, param_grid, make_search_brt_clf, seed=MODEL_SEED)

    save_run(fold_scores, model="XGBoost BRT classifier", task="classification", author="Karuna",
             split=split, use_subbasin=use_subbasin, drop_elevation=drop_elevation,
             n_features=len(features), notes="nested CV scores")
    print(fold_scores.round(3).to_string(index=False))
    return {"split": split, "subbasin": use_subbasin,
            "auc": fold_scores["auc"].mean(), "brier_skill": fold_scores["brier_skill"].mean()}

summary_clf = []
for split, use_subbasin in CONFIGS:
    start = time.time()
    summary_clf.append(run_config_clf(split, use_subbasin))
    print(f"{split} | subbasin={use_subbasin} | {(time.time() - start) / 60:.1f} min")

pd.DataFrame(summary_clf).round(3)


subfold    0    1    2    3
fold                       
0        187  195  237  416
1        109  194  230  439
2        408  258  302  175
3        186  160  224  471
Saved 4 folds to results/model_runs/karuna__xgboost-brt-classifier__classification__10km__subbasin0__noelev0.csv
 fold   auc  brier_skill  best_inner_brier  subsample  n_estimators  min_child_weight  max_depth  learning_rate
    0 0.814        0.269             0.173        0.5            75                20          3           0.03
    1 0.703        0.056             0.158        0.5            75                 5          3           0.05
    2 0.808        0.229             0.153        0.5            75                20          2           0.03
    3 0.815        0.276             0.172        0.5            75                20          3           0.03
10km | subbasin=False | 0.3 min
subfold    0    1    2    3
fold                       
0        187  195  237  416
1        109  194  230  439
2        408  2

,split,subbasin,auc,brier_skill
0,10km,False,0.785,0.207
1,10km,True,0.800,0.232
2,10km_subbasin,True,0.814,0.273
3,7.5km_subbasin,True,0.788,0.247
4,5km_subbasin,True,0.816,0.257


In [20]:
# #call the function fit_model_with_nested_cv

# outer_results, best_params_by_fold = (
# random_forest.fit_model_with_nested_cv(
#         development,
#         X_dev,
#         y_dev,
#         param_grid,
#         make_search_brt,
#         N_SUBFOLDS=N_SUBFOLDS,
#         THRESHOLD_MG_L=THRESHOLD_MG_L,
#         SEED=MODEL_SEED,
#         PRINT=True
#     )
# )

In [21]:
# def find_best_hyperparameter(all_best_params, ignore=("outer_fold", "seed")):
#     param_cols = [c for c in all_best_params.columns if c not in ignore]
#     overall = (all_best_params
#                .groupby(param_cols, dropna=False)
#                .size()
#                .reset_index(name="count")
#                .sort_values("count", ascending=False))
#     overall["frequency"] = overall["count"] / len(all_best_params)
#     return overall


In [22]:
# runs = []
# for seed in [1, 2, 3]:
#     outer_results, best_params_by_fold = random_forest.fit_model_with_nested_cv(
#         development, X_dev, y_dev, param_grid, make_search_brt,
#         N_SUBFOLDS=N_SUBFOLDS, THRESHOLD_MG_L=THRESHOLD_MG_L, SEED=seed, PRINT=False,
#     )
#     runs.append(best_params_by_fold.assign(seed=seed))

# all_best_params = pd.concat(runs, ignore_index=True)       # 3 runs × 4 outer folds = 12 rows


In [23]:
# frequency = find_best_hyperparameter(all_best_params)
# frequency


In [24]:
# param_cols = [c for c in all_best_params.columns if c not in ("outer_fold", "seed")]
# best_params = frequency[param_cols].iloc[[0]].to_dict("records")[0]
# best_params


In [25]:
# # Normal CV function (just outer folds)

# model = xgboost.XGBRegressor(learning_rate=.1, max_depth=3, n_estimators=100)
# random_forest.fit_model_with_normal_cv(development, X_dev, y_dev, model, THRESHOLD_MG_L = 10)

In [26]:
estimator=xgboost.XGBRegressor(objective="reg:squarederror", max_bin=64,
                               random_state=MODEL_SEED, n_jobs=1),


In [27]:
def cv_summary(split, use_subbasin):
    X, y, meta = load_model_data(subbasin=use_subbasin)
    d = load_split(pd.concat([meta, X], axis=1).assign(log_nitrate=y), split=split)
    tv = d[~d["holdout"]].reset_index(drop=True)
    feats, yy = list(X.columns), tv["log_nitrate"]
    pred, base = np.full(len(tv), np.nan), np.full(len(tv), np.nan)
    for train, test in splits.iter_folds(tv["fold"]):
        pred[test] = make_model().fit(tv.loc[train, feats], yy[train]).predict(tv.loc[test, feats])
        base[test] = yy[train].mean()
    return {"split": split, "subbasin": use_subbasin,
            "rmse_log": root_mean_squared_error(yy, pred), "rmse_mean": root_mean_squared_error(yy, base),
            "r2": r2_score(yy, pred), "auc": roc_auc_score(np.expm1(yy) > THRESHOLD_MG_L, pred)}

pd.DataFrame([cv_summary(s, sb) for s in SPLIT_LIST
                                for sb in [False, True]]).round(3)


,split,subbasin,rmse_log,rmse_mean,r2,auc
0,10km,False,1.238,1.238,-0.005,0.497
1,10km,True,1.238,1.238,-0.005,0.497
2,5km_subbasin,False,1.210,1.210,-0.004,0.472
3,5km_subbasin,True,1.210,1.210,-0.004,0.472
4,7.5km_subbasin,False,1.215,1.215,-0.010,0.457
5,7.5km_subbasin,True,1.215,1.215,-0.010,0.457
6,10km_subbasin,False,1.210,1.210,-0.003,0.444
7,10km_subbasin,True,1.210,1.210,-0.003,0.444


# Final Evaluation

Save for when we have chosen our final model (along with any hyperparameters) to evaluate on the test set.

In [28]:
# ============================================================
# FINAL EVALUATION ON THE HOLD-OUT FOLD
# Run this ONCE, for the final chosen model only, after all
# model choices (features, hyperparameters) are finished.
# ============================================================
RUN_FINAL_EVALUATION = False

if RUN_FINAL_EVALUATION:
    holdout_df = data[data["holdout"]].reset_index(drop=True)
    final_model = make_model()
    final_model.fit(X_tv, y_tv)                                 # all four development folds
    holdout_pred = final_model.predict(holdout_df[FEATURES])
    y_ho = holdout_df["log_nitrate"]
    print(f"Hold-out ({SPLIT}) RMSE (log): {root_mean_squared_error(y_ho, holdout_pred):.3f} "
          f"| R²: {r2_score(y_ho, holdout_pred):.3f} "
          f"| AUC: {roc_auc_score(np.expm1(y_ho) > THRESHOLD_MG_L, holdout_pred):.3f}")
